The purpose of this file is to create a table showing the data. It changes the pivot into a table we export and can display next to our graphs

In [1]:
import pandas as pd
import numpy as np
import os, csv, re
import matplotlib.pyplot as plt
from pathlib import Path
plt.close("all")
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.0f}'.format) # Turn off scientific notation
pd.set_option('display.float_format', '{:.2f}'.format) # Force display floats with 2 decimal globally

def format_headers(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace(r"[./ >#:?*\-]", "_", regex=True)
        .str.replace(r"[()']", "", regex=True)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.lower()
        .str.replace(r"_+", "_", regex=True)
        .str.strip()
        .str.strip("_")
    )
    return df


from decimal import Decimal, ROUND_HALF_UP

In [2]:
# Selects which data set we want to analyse

subject="math"
# subject="reading"

In [3]:
# Creates a folder named tables

output_dir = Path("tables")
output_dir.mkdir(parents=True, exist_ok=True)

In [4]:
# Imports our data

table = pd.read_csv(f"intermediate/01_CleanTest_{subject}.csv")
table.columns

Index(['school', 'student_grade', 'met_growth_count', 'met_growth_percent',
       'met_growth_sum', 'met_performance_count', 'met_performance_percent',
       'met_performance_sum'],
      dtype='str')

In [5]:
# Rounds the percentages to two decimal places

table["met_growth_percent"] = np.floor(table["met_growth_percent"] * (100) + 0.5) / (100)
table["met_performance_percent"] = np.floor(table["met_performance_percent"] * (100) + 0.5) / (100)

In [6]:
# Categorises the grades so K can be in front

grade_order = ["K", "1", "2", "3", "4", "5", "6", "7", "8"]

table["student_grade"] = pd.Categorical(
    table["student_grade"].astype(str),
    categories=grade_order,
    ordered=True)
table = table.sort_values("student_grade")

In [7]:
# Creates the abriviation column

table["school_abrv"] = table["school"].apply(
    lambda x: "".join(word[0] for word in x.split())
)
table["school_grade"] = (
    table["school_abrv"] + " " + table["student_grade"].astype(str)
)

In [8]:
# Reorders the table

new_order = ["school","school_grade","met_performance_sum","met_performance_count","met_performance_percent","met_growth_sum","met_growth_count",
             "met_growth_percent"]
table = table[new_order]

In [9]:
# Renames the labels of the table

rename_dict = {
    "school":"School",
    "school_grade":" ",
    "met_growth_count":"Eligible Students for Growth (n)",
    "met_growth_percent":"Students Met 100% Annual Typical Growth (%)",
    "met_growth_sum":"Students Met 100% Annual Typical Growth (n)",
    "met_performance_count":"Eligible Students for Performance (n)",
    "met_performance_percent":"Students Met Grade Level Performance (%)",
    "met_performance_sum":"Students Met Grade Level Performance (n)"
}

a_table = table.rename(columns = rename_dict).copy()

a_table.head(10)

,School,,Students Met Grade Level Performance (n),Eligible Students for Performance (n),Students Met Grade Level Performance (%),Students Met 100% Annual Typical Growth (n),Eligible Students for Growth (n),Students Met 100% Annual Typical Growth (%)
152,Summit Valley School District,SVSD K,235,464,0.51,204.00,408,0.50
44,Central High,CH K,20,31,0.65,18.00,30,0.60
26,Canyon Ridge Elementary,CRE K,14,30,0.47,12.00,23,0.52
53,Eagle View Elementary,EVE K,15,28,0.54,17.00,24,0.71
62,East High,EH K,13,24,0.54,6.00,19,0.32
71,Falcon Ridge Elementary,FRE K,18,36,0.50,15.00,31,0.48
80,Foothills Elementary,FE K,13,28,0.46,13.00,25,0.52
17,Bear Creek Elementary,BCE K,12,34,0.35,15.00,31,0.48
89,Grand Mesa Elementary,GME K,15,29,0.52,11.00,23,0.48
98,Heritage Middle,HM K,12,19,0.63,8.00,18,0.44


In [10]:
# Exports the table to excel

a_table.to_excel(f"tables/{subject}_table.xlsx",index=False)
print("done")

done
